# Aggregate the single-cell profiles to the well level
This notebook is not run as a large amount of RAM is needed to run it. It is provided for reference only.

In [1]:
import os
import pathlib

import pandas as pd
from pycytominer import aggregate
from pycytominer.cyto_utils import output
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

In [ ]:
# load in platemap file as a pandas dataframe
platemap_path = pathlib.Path(
    f"{root_dir}/Wave2_data/0.download_data/platemap/platemap.csv"
).resolve()

image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(f"{image_base_dir}/processed_data/").resolve(strict=True)
normalized_profiles_path = pathlib.Path(
    f"{image_base_dir}/8.normalized_profiles/normalized_profiles.parquet"
).resolve(strict=True)
chammi_profiles_path = pathlib.Path(
    f"{image_base_dir}/7a.CHAMMI75_extracted_features/CHAMMI75_combined_sc_profiles.parquet"
).resolve(strict=True)

harmonized_profiles_path = pathlib.Path(
    f"{image_base_dir}/9.harmonized_profiles/harmonized_profiles.parquet"
).resolve()
harmonized_profiles_path.parent.mkdir(parents=True, exist_ok=True)

In [3]:
norm_CP_df = pd.read_parquet(normalized_profiles_path)
chammi75_df = pd.read_parquet(chammi_profiles_path)
print("Shape of profiles:")
print(f"Normalized CP: {norm_CP_df.shape[0]}")
print(f"CHAMMI75: {chammi75_df.shape[0]}")

Shape of profiles:
Normalized CP: 4627
CHAMMI75: 1136


In [ ]:
merged_df = pd.merge(
    norm_CP_df,
    chammi75_df,
    on=[
        "Metadata_Well_FOV",
        "Metadata_Time",
        "Metadata_Nuclei_Number_Object_Number",
        "Metadata_Cells_AreaShape_Center_X",
        "Metadata_Cells_AreaShape_Center_Y",
    ],
    how="left",
)

if norm_CP_df.shape[0] + chammi75_df.shape[0] != merged_df.shape[0]:
    print(
        "Warning: The number of rows in the merged dataframe does not equal the sum of the input dataframes. Please check for duplicates or missing values in the key columns."
    )
    print(f"Number of rows in normalized CP dataframe: {norm_CP_df.shape[0]}")
    print(f"Number of rows in CHAMMI75 dataframe: {chammi75_df.shape[0]}")
    print(f"Number of rows in merged dataframe: {merged_df.shape[0]}")

if merged_df.isnull().any().any():
    print(
        "Warning: There are missing values in the merged dataframe. Please check for mismatches in the key columns between the two dataframes."
    )

merged_df.to_parquet(harmonized_profiles_path, index=False)
merged_df.head()

Number of rows in normalized CP dataframe: 4627
Number of rows in CHAMMI75 dataframe: 1136
Number of rows in merged dataframe: 4627
Merged profiles: 4627


,Metadata_Well,Metadata_Inducer,Metadata_Inducer_dose,Metadata_Inhibitor,Metadata_Inhibitor_dose,Metadata_Time,Metadata_Well_FOV,Metadata_FOV,Metadata_Well_FOV_Time,Metadata_ImageNumber,...,Nucleocentric_SYTOXGreen_CHAMMI75_Feature90,Nucleocentric_SYTOXGreen_CHAMMI75_Feature91,Nucleocentric_SYTOXGreen_CHAMMI75_Feature92,Nucleocentric_SYTOXGreen_CHAMMI75_Feature93,Nucleocentric_SYTOXGreen_CHAMMI75_Feature94,Nucleocentric_SYTOXGreen_CHAMMI75_Feature95,Nucleocentric_SYTOXGreen_CHAMMI75_Feature96,Nucleocentric_SYTOXGreen_CHAMMI75_Feature97,Nucleocentric_SYTOXGreen_CHAMMI75_Feature98,Nucleocentric_SYTOXGreen_CHAMMI75_Feature99
0,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,1.023240,1.254434,-5.106654,3.973393,4.420975,2.096377,-1.455441,1.754054,0.756196,2.069703
1,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,1.118941,1.750928,-4.207298,4.641288,5.062917,1.886621,-1.790866,2.684594,1.007277,1.772836
2,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,1.056204,1.504928,-4.435646,4.207762,3.852463,1.584646,-0.776207,1.837654,0.255680,0.995764
3,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,1.022551,0.869031,-5.359560,4.268159,4.884048,2.598779,-1.734051,1.474915,0.430418,2.043305
4,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,1.193125,1.376956,-5.014697,5.400067,4.056452,2.430958,-1.970731,1.144398,-0.105689,1.925226
